# 01 – Datenbeschaffung und Datenqualität

**Ziel:** Day-Ahead-Strompreise sowie Wind-, Solar- und Lastdaten für die deutsche
Gebotszone **DE-LU** (2023–2025) über die öffentliche
[Energy-Charts API](https://api.energy-charts.info/) (Fraunhofer ISE) beziehen und
in eine saubere, lückenlose Stundenreihe überführen.

Die eigentliche Logik liegt im Paket `src/energy_analytics` und wird über
`python -m energy_analytics.pipeline` ausgeführt. Dieses Notebook dokumentiert
die Entscheidungen, die dabei getroffen wurden:

1. Aufbau der API-Antworten
2. Zeitzonen und Zeitumstellung
3. Wechsel der Preisauflösung von 60 auf 15 Minuten (Oktober 2025)
4. Qualitätsprüfung des zusammengeführten Datensatzes

In [1]:
import sys
from pathlib import Path

# Projektpaket importierbar machen (alternativ: pip install -e .)
sys.path.insert(0, str(Path.cwd().parent / "src"))

In [2]:
import pandas as pd

from energy_analytics import config
from energy_analytics.extract import EnergyChartsClient
from energy_analytics.transform import build_hourly_dataset, quality_report, read_raw_csv

## 1. Aufbau der API-Antworten

Beide Endpunkte liefern Unix-Zeitstempel (UTC) und parallele Wertelisten. Zur
Veranschaulichung wird ein einzelner Tag abgefragt. Die API ist rate-limitiert –
der Client wiederholt Anfragen bei HTTP 429 automatisch mit exponentiellem Backoff.

In [3]:
client = EnergyChartsClient()
sample = client._get("price", {"bzn": config.BIDDING_ZONE, "start": "2025-06-01", "end": "2025-06-01"})

print("Schlüssel:", list(sample))
print("Einheit:  ", sample["unit"])
print("Lizenz:   ", sample["license_info"])
print("Werte:    ", len(sample["price"]))

Schlüssel: ['license_info', 'unix_seconds', 'price', 'unit', 'deprecated']
Einheit:   EUR / MWh
Lizenz:    CC BY 4.0 (creativecommons.org/licenses/by/4.0) from Bundesnetzagentur | SMARD.de
Werte:     24


## 2. Rohdaten laden

Die Pipeline speichert die Rohdaten unverändert unter `data/raw/`. Abgerufen
werden zusätzlich 8 Tage **vor** dem Untersuchungszeitraum, damit Lag-Features
(bis 168 Stunden) bereits ab dem 01.01.2023 vollständig berechnet werden können.

In [4]:
prices = read_raw_csv(config.RAW_PRICE_FILE)
power = read_raw_csv(config.RAW_POWER_FILE)

print(f"Preise:  {len(prices):>7,} Zeilen  {prices['timestamp'].min()}  bis  {prices['timestamp'].max()}")
print(f"Erzeugung: {len(power):>5,} Zeilen  {power['timestamp'].min()}  bis  {power['timestamp'].max()}")
power.head()

Preise:   33,219 Zeilen  2022-12-23 23:00:00+00:00  bis  2026-01-01 22:45:00+00:00
Erzeugung: 106,080 Zeilen  2022-12-23 23:00:00+00:00  bis  2026-01-01 22:45:00+00:00


,timestamp,wind_offshore,wind_onshore,solar,load
0,2022-12-23 23:00:00+00:00,2318.4,19312.7,0.0,41081.7
1,2022-12-23 23:15:00+00:00,2292.9,19465.3,0.0,39803.0
2,2022-12-23 23:30:00+00:00,2245.6,19708.5,0.0,38795.2
3,2022-12-23 23:45:00+00:00,2135.0,19287.1,0.0,38565.5
4,2022-12-24 00:00:00+00:00,1996.4,19263.6,0.0,37615.2


## 3. Zeitliche Auflösung

Erzeugung und Last liegen durchgehend viertelstündlich vor. Bei den Preisen ändert
sich die Auflösung: Seit dem **01.10.2025** wird der Day-Ahead-Markt in
15-Minuten-Produkten gehandelt.

In [5]:
resolution = (
    prices.assign(
        month=prices["timestamp"].dt.tz_convert(config.LOCAL_TZ).dt.strftime("%Y-%m"),
        step=prices["timestamp"].diff(),
    )
    .groupby("month")["step"]
    .agg(lambda s: s.mode().iloc[0])
)
resolution.loc["2025-08":"2025-12"]

month
2025-08   0 days 01:00:00
2025-09   0 days 01:00:00
2025-10   0 days 00:15:00
2025-11   0 days 00:15:00
2025-12   0 days 00:15:00
Name: step, dtype: timedelta64[us]

**Entscheidung:** Alle Reihen werden auf **Stundenmittelwerte** aggregiert. Der
Mittelwert von vier Viertelstundenpreisen entspricht dem mengengleich gewichteten
Stundenpreis und hält die Zeitreihe über den gesamten Zeitraum vergleichbar.

Die Aggregation erfolgt in **UTC**. In Ortszeit hätte der Tag der Umstellung auf
Sommerzeit 23 Stunden, der Tag der Rückumstellung 25 Stunden – in UTC entstehen
dadurch weder Lücken noch Duplikate. Kalendermerkmale (Stunde, Wochentag …) werden
erst im Feature Engineering in **deutscher Ortszeit** abgeleitet.

## 4. Zusammenführen und Qualitätsprüfung

In [6]:
hourly = build_hourly_dataset(prices, power, include_history=False)
print(quality_report(hourly))

Zeilen:              26,304
Zeitraum (lokal):    2023-01-01 00:00 bis 2025-12-31 23:00
Fehlende Stunden:    0
Doppelte Zeitpunkte: 0
Fehlende Werte:      0
Status:              OK


In [7]:
hourly.describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
price,26304.0,87.7,51.3,-500.0,65.1,90.1,113.0,936.3
wind_offshore,26304.0,2862.0,1931.8,0.0,1126.6,2612.1,4472.9,8237.8
wind_onshore,26304.0,12608.1,9723.7,45.8,4850.7,10014.9,18262.4,47795.3
solar,26304.0,6985.2,10656.8,0.0,0.6,227.3,11262.8,49713.9
load,26304.0,52832.4,9131.1,30902.7,45329.0,52882.6,59712.7,75635.2
wind,26304.0,15470.1,10975.6,45.8,6444.7,12909.4,22494.9,52173.8


**Ergebnis:** 26.304 Stunden (3 × 8.760 + 24 h Schaltjahr 2024), keine fehlenden
Stunden, keine Duplikate, keine fehlenden Werte. Auffällig sind die Extremwerte des
Preises (−500 bis 936 EUR/MWh) – sie sind real (z. B. −500 EUR/MWh am Sonntag, 02.07.2023, bei
sehr hoher Solar- und Windeinspeisung) und werden **nicht** als
Ausreißer entfernt, sondern in der EDA gezielt untersucht.